# 17 - Collect CartPole with Hidden Velocities

Collect ordinary CartPole episodes while exposing only cart position and pole angle. The
environment wrapper removes both velocities before observations reach the event log. A
policy must use observation history to infer motion.

Actions are `0` (push left) and `1` (push right). This collector samples them randomly.
CartPole-v1 ends when the pole falls or the cart leaves the track, and truncates at its
native 500-step limit. Example 18 trains on these events.


In [ ]:
import gymnasium as gym
from mouse_gym import EnvConfig, RolloutTracker, make_group_env

from mouse_core.data import EventLog, push_stores_to_hub

# Repositories
# Hugging Face dataset repo for push_stores_to_hub
DATASET_ID = "mouse-example-cartpole-dataset"

# Environment
# number of environment streams in the GroupEnv
NUM_ENVS = 4

# Collection
# environment outputs collected per stream
STEPS_PER_ENV = 1000

## Build Environment

`TransformObservation` selects position and angle and exposes a two-dimensional
observation space. Each mouse-gym stream resets after an ordinary CartPole episode.


In [ ]:
def make_cartpole():
    cartpole = gym.make("CartPole-v1")
    bounds = cartpole.observation_space
    assert isinstance(bounds, gym.spaces.Box)
    return gym.wrappers.TransformObservation(
        cartpole,
        func=lambda observation: observation[[0, 2]],
        observation_space=gym.spaces.Box(
            low=bounds.low[[0, 2]],
            high=bounds.high[[0, 2]],
            dtype=bounds.dtype,
        ),
    )


configs = [
    EnvConfig(
        name=f"cartpole_{i}",
        seed=i,
        seed_mode="stream",
        env_fn=make_cartpole,
    )
    for i in range(NUM_ENVS)
]
env = make_group_env(configs)
logs = [EventLog(name=name) for name in env.names]
trackers = [RolloutTracker() for _ in logs]

## Record Events

Record each requested action before stepping. Reset calls receive `None`. Each
environment event records `cart_position` and `pole_angle` as named scalar fields, plus
reward and episode status.


In [ ]:
for step in range(STEPS_PER_ENV):
    for log, tracker in zip(logs, trackers):
        episode_start = tracker.output is None or tracker.ended
        if episode_start:
            log.event(
                provider="coordinator",
                episode_index=tracker.next_episode_index,
                step_index=(0 if episode_start else tracker.step_index + 1),
                payload={
                    "episode_start": True,
                },
            )

    actions: list[int | None] = [None] * len(logs)
    for i, (log, tracker) in enumerate(zip(logs, trackers)):
        if tracker.action_requested:
            actions[i] = int(env.action_space.spaces[i].sample())
            log.event(
                provider="agent",
                episode_index=tracker.next_episode_index,
                step_index=tracker.step_index + 1,
                payload={
                    "action": actions[i],
                },
            )
    outputs = env.step(tuple(actions))
    for i, (tracker, output) in enumerate(zip(trackers, outputs)):
        tracker.update(output=output)
        log = logs[i]

        log.event(
            provider="environment",
            episode_index=tracker.episode_index,
            step_index=tracker.step_index,
            payload={
                "cart_position": tracker.observation[0],
                "pole_angle": tracker.observation[1],
                **({"reward": tracker.reward} if tracker.reward != 0 else {}),
                **({"terminated": True} if tracker.terminated else {}),
                **({"truncated": True} if tracker.truncated else {}),
                **({"action_requested": True} if tracker.action_requested else {}),
            },
        )
env.close()
stores = [log.to_store() for log in logs]

## Save To The Hub

`push_stores_to_hub` uploads every datastore to one Hugging Face dataset repository.
Each store is one config. `18_train_offline_cartpole_dqn.ipynb` reloads them with
`load_stores_from_hub`.


In [ ]:
url = push_stores_to_hub(
    stores=stores, repo_id=DATASET_ID, split="train", private=False
)
print(f"Pushed to {url}")